# `gen_spectra` — one primitive, two knobs

Sub-band **BaF X $^2\Sigma^+$(N=0,+) $\rightarrow$ A $^2\Pi_{1/2}$(J=1/2,$-$)**
(native BaF database constants: `molecule_name='BaF'`, isotope 138).

The line strength $S=\sum_{M'',M',p}|\langle g|T^1_p(d)|e\rangle|^2$ is the only
independent quantity. The physical observable is chosen by two orthogonal
knobs, **not** by whether the states were built M-resolved:

| `initial` | `initial_reduction` | observable |
|---|---|---|
| (any) | `sum` *(default)* | line strength $S$ = LIF excitation signal |
| `excited` | `average` | emission branching $S/(2F'{+}1)$ = `branching_ratios` |
| `ground` | `average` | per-molecule absorption $\tilde\sigma\propto S/(2F''{+}1)$ |

`cross_section` and `branching` are the *same* operation (÷ initial-state
degeneracy) with the initial state swapped. no-M and M='all' give the same
observable (the sanity cell asserts it).

In [ ]:
from config_path import add_to_sys_path
add_to_sys_path()
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from Energy_Levels import MoleculeLevels
import gen_spectra as gs
pd.set_option('display.width', 160)

In [ ]:
def build(elec, Nl, M):
    P = [1/2] if elec == 'X' else [1/2, 3/2]
    return MoleculeLevels.initialize_state(
        molecule_name='BaF', elec_state=elec, vib_state=0,
        N_list=np.array(Nl), fermion_or_boson='boson', M_sublevels=M,
        I_nuclei=[0, 1/2], isotope=138, round=8, params=None, P_values=P)

g  = build('X', [0, 1], 'none'); e  = build('A', [1, 2], 'none')
gM = build('X', [0, 1], 'all');  eM = build('A', [1, 2], 'all')
for s in (g, e, gM, eM):
    s.eigensystem(0, 0)

gi,  ei  = g.select_q({'N': 0}),  e.select_q({'J': 0.5}, parity='-')
giM, eiM = gM.select_q({'N': 0}), eM.select_q({'J': 0.5}, parity='-')
print('no-M sizes  g=%d e=%d | M=all sizes  g=%d e=%d'
      % (g.size, e.size, gM.size, eM.size))

## The three observables (no-M basis), grouped by (F'', F')

In [ ]:
def grp(L):
    return (L.groupby(['g_F', 'e_F'], as_index=False)
             .agg(freq=('freq', 'mean'), s=('strength', 'sum'))
             .sort_values(['g_F', 'e_F']).reset_index(drop=True))

org = e.parameters['Origin']
S  = grp(gs.line_list(g, e, gi, ei, origin=org))                                            # default = line strength
br = grp(gs.line_list(g, e, gi, ei, origin=org, initial='excited', initial_reduction='average'))
cx = grp(gs.line_list(g, e, gi, ei, origin=org, initial='ground',  initial_reduction='average'))
T = S[['g_F', 'e_F']].copy()
T['line_strength_S']            = S['s']
T['branching_S_over_2Fp1']      = br['s']
T['cross_section_S_over_2Fpp1'] = cx['s']
T

## Sanity: no-M and M='all' compute the *same* observable

In [ ]:
orgM = eM.parameters['Origin']
S_M  = grp(gs.line_list(gM, eM, giM, eiM, origin=orgM))
br_M = grp(gs.line_list(gM, eM, giM, eiM, origin=orgM, initial='excited', initial_reduction='average'))
ok1 = np.allclose(S['s'].values,  S_M['s'].values,  rtol=1e-6, atol=1e-9)
ok2 = np.allclose(br['s'].values, br_M['s'].values, rtol=1e-6, atol=1e-9)
ok3 = np.allclose(S['s'].values,  br['s'].values * (2 * S['e_F'].values + 1),
                  rtol=1e-6, atol=1e-9)
print('line strength   no-M == M=all :', ok1)
print('branching       no-M == M=all :', ok2)
print("S == branching x (2F'+1)      :", ok3)
assert ok1 and ok2 and ok3, 'cross-base / observable identity failed'

## The LIF spectrum (line strength, the default)

Overlay a measured scan with `plot_spectrum(..., experimental=dict(freq=,
signal=, err=, tweak=, yscale=))` — see `BaF_spectrum_plot.py`.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.2))
gs.plot_spectrum(Ground=g, Excited=e, g_idx=gi, e_idx=ei, origin=org, ax=ax,
                 sticks=True,
                 broaden_kw=dict(shape='voigt', fwhm=10.0, lorentz_fwhm=3.5),
                 label='line strength (LIF)')
ax.set_xlabel('frequency (MHz)'); ax.set_ylabel('norm. signal')
ax.set_title(r'BaF  X(N=0,+) $\rightarrow$ A$^2\Pi_{1/2}$(J=1/2,$-$)  — line strength')
plt.tight_layout(); plt.show()